# Z-scores across eras, and what they hide

*A method chapter. Structure: Question → Intuition → Math → Code → Assumptions →
How it breaks.*

## 1. Question

How do you compare a striker from 2003 with one from 2024, when the game they
played was not the same game?

Concretely: in some seasons goals are plentiful and in others they are scarce.
A player who scored 0.8 goals per 90 in a low-scoring season may have been more
dominant than one who scored 0.9 in a high-scoring one. Raw rates cannot see
this. We need a number that means "how far ahead of your peers were you".

## 2. Intuition

Stop measuring in goals and start measuring in **peers**.

If the typical player in your league scored 0.30 goals per 90 that season, and
the spread among players was about 0.15, then scoring 0.60 puts you two spreads
above typical. Do the same arithmetic in a different season with different
numbers and "two spreads above typical" still means the same thing: you were
unusually good by the standards of the football around you.

The unit travels across eras even though goals do not. That is the whole idea.

## 3. Math

For player $i$ in season $s$, with rate $x_{is}$:

$$z_{is} = \frac{x_{is} - \mu_s}{\sigma_s}$$

where $\mu_s$ and $\sigma_s$ are the mean and standard deviation of the rate
across all players in season $s$.

**Choice: population, not sample standard deviation** ($\sigma$ with $N$ in the
denominator, `ddof=0`). A season is not a sample drawn from some larger
population of that season — it is every player who played it. The whole
population is in hand, so the population formula is the correct one. With ~500
players per season the numerical difference is negligible, but the reasoning
matters more than the digits.

**The small-sample problem.** A player with 200 minutes who happened to score
twice gets a spectacular rate and therefore a spectacular $z$. This is noise, not
ability. We discount it by shrinking toward zero in proportion to playing time:

$$\hat{z}_{is} = z_{is} \cdot \frac{m_{is}}{m_{is} + m_0}$$

where $m_{is}$ is minutes played and $m_0$ is a prior strength, also in minutes.
The weight $m/(m + m_0)$ runs from 0 (no minutes, no claim) to 1 (many minutes,
take the number at face value), passing through exactly $1/2$ at $m = m_0$.

We use $m_0 = 900$ — ten full matches. A player with ten matches is credited with
half of what their raw score claims, which is about right for how much you should
believe ten games.

## 4. Code

Small enough to check by hand.

In [ ]:
import numpy as np
import pandas as pd

from gambeta import level

toy = pd.DataFrame(
    {
        "league": ["L"] * 4,
        "season": ["0001"] * 4,
        "player_id": list("abcd"),
        "ga_p90": [0.2, 0.4, 0.6, 0.8],
        "minutes": [3000, 3000, 3000, 3000],
    }
)

out = level.zscore(toy, ["ga_p90"])
out[["player_id", "ga_p90", "ga_p90_z"]]

Check by hand: the mean of 0.2, 0.4, 0.6, 0.8 is 0.5. The population standard
deviation is

$$\sigma = \sqrt{\tfrac{1}{4}\left((0.3)^2 + (0.1)^2 + (0.1)^2 + (0.3)^2\right)}
= \sqrt{0.05} \approx 0.2236$$

So player `d` scores $(0.8 - 0.5)/0.2236 \approx 1.342$. Confirm:

In [ ]:
expected = (0.8 - 0.5) / np.sqrt(0.05)
actual = out.loc[out["player_id"] == "d", "ga_p90_z"].item()
print(f"by hand: {expected:.6f}")
print(f"gambeta: {actual:.6f}")
assert np.isclose(expected, actual)

Now shrinkage. The same score, held by players with very different amounts of
football behind it:

In [ ]:
minutes = np.array([90.0, 450.0, 900.0, 1800.0, 3600.0])
shrunk = level.shrink(np.full(5, 2.0), minutes, prior_minutes=900.0)

pd.DataFrame(
    {
        "minutes": minutes.astype(int),
        "raw z": 2.0,
        "weight": (minutes / (minutes + 900.0)).round(3),
        "shrunk z": shrunk.round(3),
    }
)

One match of brilliance retains a tenth of its claim. Twenty matches retain
half. Forty matches retain four-fifths. Nothing is thrown away, and nothing
small is believed.

## 5. Assumptions

Each of these could be false, and each would bite differently.

1. **Within-season distributions are comparable in shape.** A z-score of 2.0 means
   the same thing in 2004 and 2024 only if both seasons' rate distributions are
   similarly shaped. If one season is heavily skewed and the other is not, the
   same z corresponds to different percentiles.

2. **The competitive spread is stable.** Standardising divides by the spread, so a
   season where players are unusually similar inflates everyone's z-scores. If the
   league grew more unequal over time, this method partly measures inequality
   rather than ability.

3. **Minutes are a good proxy for sample size.** Shrinkage assumes 900 minutes of
   a defender and 900 minutes of a striker carry the same evidential weight for
   attacking output. They plainly do not.

4. **The population is the right comparison set.** We standardise against all
   players, including goalkeepers and centre-backs who are not trying to score.
   That drags the mean down and inflates every attacker's z-score. Standardising
   within position would be more defensible; it is not done here, and it is a
   real weakness rather than a rounding detail.

## 6. How it breaks

The failure worth understanding is that **a single outlier suppresses everyone,
including the outlier.**

The standard deviation is in the denominator. One extraordinary season inflates
it, which shrinks every z-score in that season — so a historically great campaign
can make itself, and everyone around it, look more ordinary.

Watch it happen.

In [ ]:
normal = pd.DataFrame(
    {
        "league": ["L"] * 5,
        "season": ["0001"] * 5,
        "player_id": list("abcde"),
        "ga_p90": [0.20, 0.30, 0.40, 0.50, 0.90],
        "minutes": [3000] * 5,
    }
)

# Same league, except the best player has a genuinely historic season.
outlier = normal.copy()
outlier.loc[outlier["player_id"] == "e", "ga_p90"] = 2.50

a = level.zscore(normal, ["ga_p90"]).set_index("player_id")["ga_p90_z"]
b = level.zscore(outlier, ["ga_p90"]).set_index("player_id")["ga_p90_z"]

pd.DataFrame({"z (normal season)": a.round(3), "z (with an outlier)": b.round(3)})

Player `d` scored exactly 0.50 goals per 90 in both worlds. Their football did
not change at all. But their z-score *fell*, because someone else had a
spectacular year and widened the yardstick.

That is not a bug in the arithmetic — it is what standardisation means. But it
has a real consequence for this project: **a season containing a historic
individual campaign will systematically under-rate everyone in it, including the
player who produced it.**

### What to do about it

Three options, in increasing order of effort:

- **Report it.** State that z-scores are relative and that outlier seasons
  compress the field. Cheapest, and better than silence.
- **Use a robust scale.** Replace the standard deviation with the median absolute
  deviation, which a single extreme value barely moves.
- **Model it properly.** A hierarchical model estimates season effects and player
  ability jointly, so an outlier is explained as an unusual player rather than
  absorbed into the season's yardstick. This is the Phase 3 direction.

Phase 1 does the first. Knowing which one you are doing, and why, is the
difference between using a method and trusting it.